In [15]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone


In [16]:
RANDOM_STATE = 42
CV = 5

In [17]:
df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

In [18]:
df_summary = pd.DataFrame({'null': df_train.isna().sum(), 'unique': df_train.nunique(), 'type': df_train.dtypes, 'sample': df_train.iloc[0]})
df_summary

,null,unique,type,sample
PassengerId,0,891,int64,1
Survived,0,2,int64,0
Pclass,0,3,int64,3
Name,0,891,str,"Braund, Mr. Owen Harris"
Sex,0,2,str,male
Age,177,88,float64,22.0
SibSp,0,7,int64,1
Parch,0,7,int64,0
Ticket,0,681,str,A/5 21171
Fare,0,248,float64,7.25


In [19]:
for col in df_train.drop(columns=['PassengerId']).columns:
    print(col, df_train[col].unique()[0:7])

Survived [0 1]
Pclass [3 1 2]
Name <StringArray>
[                            'Braund, Mr. Owen Harris',
 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
                              'Heikkinen, Miss. Laina',
        'Futrelle, Mrs. Jacques Heath (Lily May Peel)',
                            'Allen, Mr. William Henry',
                                    'Moran, Mr. James',
                             'McCarthy, Mr. Timothy J']
Length: 7, dtype: str
Sex <StringArray>
['male', 'female']
Length: 2, dtype: str
Age [22. 38. 26. 35. nan 54.  2.]
SibSp [1 0 3 4 2 5 8]
Parch [0 1 2 5 3 4 6]
Ticket <StringArray>
[       'A/5 21171',         'PC 17599', 'STON/O2. 3101282',
           '113803',           '373450',           '330877',
            '17463']
Length: 7, dtype: str
Fare [ 7.25   71.2833  7.925  53.1     8.05    8.4583 51.8625]
Cabin <StringArray>
[nan, 'C85', 'C123', 'E46', 'G6', 'C103', 'D56']
Length: 7, dtype: str
Embarked <StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype

In [20]:
df_train['Cabin'].unique()

<StringArray>
[          nan,         'C85',        'C123',         'E46',          'G6',
        'C103',         'D56',          'A6', 'C23 C25 C27',         'B78',
 ...
        'B102',         'B69',         'E49',         'C47',         'D28',
         'E17',         'A24',         'C50',         'B42',        'C148']
Length: 148, dtype: str

In [21]:
num_cols = ['Age', 'FamilySize', 'SibSp', 'Parch', 'Fare']
cat_cols = ['Sex', 'Embarked', 'Title']
ord_cols = ['Pclass', 'IsAlone', 'GroupSize']
none_cat_cols = ['Deck']

In [22]:
counts = pd.concat([df_train['Ticket'], df_test['Ticket']]).value_counts()
def add_features(df):
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['GroupSize'] = df['Ticket'].map(counts)
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Title'] = df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()
    df['Deck'] = df['Cabin'].str[0]
    df = df.drop(columns=['Ticket', 'Cabin'])
    return df

In [23]:
df_train = add_features(df_train)
df_test = add_features(df_test)

In [24]:
X_train = df_train.drop(columns=['PassengerId', 'Survived'])
y_train = df_train['Survived']
X_test = df_test.drop(columns=['PassengerId'])
test_id = df_test['PassengerId']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [25]:

def get_dummy_prediction(X, y):
    dummy = DummyClassifier(strategy='constant', constant=0, random_state=42)
    score = -cross_val_score(estimator=dummy, X=X, y=y, scoring='accuracy', cv=cv)
    return score

def build_preprocessor(num_cols, cat_cols, none_cat_cols, ord_cols):
    num_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
    none_cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='constant', fill_value='None')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
    ord_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent'))])
    preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_cols), ('cat', cat_pipeline, cat_cols), ('none_cat', none_cat_pipeline, none_cat_cols), ('ord', ord_pipeline, ord_cols)])
    return preprocessor

preprocessor = build_preprocessor(num_cols=num_cols, cat_cols=cat_cols, none_cat_cols=none_cat_cols, ord_cols=ord_cols)

def build_pipeline(preprocessor, model):
    full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    return full_pipeline

models = {
    'xgboost': XGBClassifier(random_state=RANDOM_STATE),
    'catboost': CatBoostClassifier(random_state=RANDOM_STATE, allow_writing_files=False, verbose=0),
    'logistic': LogisticRegression(random_state=RANDOM_STATE, max_iter=10000)
}

param_distributions = {
    'xgboost': {'model__max_depth': [2,3,4,5], 'model__learning_rate': [0.01, 0.03, 0.1], 'model__n_estimators': [100,300,600], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.6, 0.8, 1.0], 'model__min_child_weight': [1,3,5]}
    ,'catboost': {'model__max_depth': [3,4,5,6], 'model__learning_rate': [0.03, 0.05, 0.1], 'model__iterations': [300,500,1000], 'model__l2_leaf_reg': [1,3,5,10]},
    'logistic': {'model__C': [0.01, 0.1, 1, 10]}
}

family = {
    'xgboost': 'trees',
    'catboost': 'trees',
    'logistic': 'linear'
}
def evaluate_model(model_name, model):
    pipeline = build_pipeline(preprocessor=preprocessor, model=model)
    accuracy = cross_val_score(estimator=pipeline, X=X_train, y=y_train, cv=cv, scoring='accuracy')
    return {
        'model': model_name,
        'accuracy': accuracy
    }
    
def select_best_result(results):
    if results.empty:
        raise ValueError('Some error happened')
    else:
        best_result = results.loc[results['validation_accuracy'].idxmax()]
        return best_result

def tune_model(model_name, model, param_distributions, n_iter=30):
    pipeline = build_pipeline(preprocessor=preprocessor, model=model)
    search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_distributions, n_iter=n_iter, scoring='accuracy', n_jobs=-1, cv=cv, return_train_score=True, refit=False, random_state=RANDOM_STATE, error_score='raise')
    search.fit(X_train, y_train)
    results = pd.DataFrame(search.cv_results_)
    results['model'] = model_name
    results['validation_accuracy'] = results['mean_test_score']
    results['train_accuracy'] = results['mean_train_score']
    best_result = select_best_result(results=results)
    return search, results, best_result

searches = {}
tuning_results = []
best_results = []
for model_name, model in models.items():
    search, results, best_result = tune_model(model_name=model_name, model=model, param_distributions=param_distributions[model_name])
    searches[model_name] = search
    tuning_results.append(results)
    best_results.append(best_result)
    
all_tuning_results = pd.concat(tuning_results, ignore_index=True)
best_results_df = pd.DataFrame(best_results).sort_values(by='validation_accuracy', ascending=False)

    

c:\Users\yazid\anaconda3\envs\ds_env\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 4 is smaller than n_iter=30. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [26]:
best_results_df['family'] = best_results_df['model'].map(family)
best_model_per_family = best_results_df.loc[best_results_df.groupby('family')['validation_accuracy'].idxmax()]
best_model_per_family


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__subsample,param_model__n_estimators,param_model__min_child_weight,param_model__max_depth,param_model__learning_rate,param_model__colsample_bytree,...,split4_train_score,mean_train_score,std_train_score,model,validation_accuracy,train_accuracy,param_model__l2_leaf_reg,param_model__iterations,param_model__C,family
3,0.192509,0.042439,0.034112,0.011066,NaN,NaN,NaN,NaN,NaN,NaN,...,0.840112,0.839785,0.003662,logistic,0.826018,0.839785,NaN,NaN,10.0,linear
13,2.536146,0.038191,0.046793,0.005203,NaN,NaN,NaN,4.0,0.03,NaN,...,0.914446,0.918069,0.002622,catboost,0.852960,0.918069,10.0,1000.0,NaN,trees


In [ ]:


def make_submission(best_models):
    predictions_proba = []
    for _, results in best_models.iterrows():
        model = clone(models[results['model']])
        pipeline = build_pipeline(preprocessor=preprocessor, model=model)
        pipeline.set_params(**results['params'])
        pipeline.fit(X_train, y_train)
        prediction_proba = pipeline.predict_proba(X_test)[:,1]
        predictions_proba.append(prediction_proba)
    avg_proba = np.mean(predictions_proba, axis=0)
    ensemble_prediction = (avg_proba > 0.5).astype(int)
    submission = pd.DataFrame({'PassengerId': test_id, 'Survived': ensemble_prediction})
    submission.to_csv('../submission2.csv', index=False)
    

In [28]:
make_submission(best_models=best_model_per_family)

[array([0.08683204, 0.5735225 , 0.08977559, 0.07802109, 0.64985928,
       0.10932033, 0.66847414, 0.09989468, 0.85790186, 0.04245063,
       0.07586451, 0.20795527, 0.96941954, 0.05432189, 0.97488865,
       0.90662946, 0.17646967, 0.12565379, 0.53556857, 0.73662974,
       0.20266098, 0.78031065, 0.94490005, 0.45923662, 0.92565851,
       0.03454761, 0.96457101, 0.1210321 , 0.29906691, 0.04929287,
       0.07680964, 0.08565418, 0.50378269, 0.54106406, 0.37035447,
       0.13369318, 0.60514517, 0.65201922, 0.08237502, 0.15064722,
       0.08130661, 0.58063738, 0.05363173, 0.87254748, 0.96568421,
       0.0820853 , 0.35307023, 0.10267013, 0.97447818, 0.63505706,
       0.32819919, 0.21820163, 0.51560927, 0.62555613, 0.2136027 ,
       0.30678607, 0.06300916, 0.17022243, 0.0630082 , 0.98091041,
       0.10097523, 0.14374615, 0.09838596, 0.7169752 , 0.95034604,
       0.89025016, 0.73959666, 0.22571139, 0.39869618, 0.68834064,
       0.70528067, 0.09108769, 0.59824258, 0.66287002, 0.9615